In [24]:
import fitz  # PyMuPDF

def read_pdf(pdf_path: str, debug: bool = False) -> dict:
    """
    Membaca hanya halaman pertama dari dokumen PDF.

    Alur:
    1. Buka file PDF.
    2. Ekstrak teks dari halaman pertama (OCR jika kosong).
    3. Simpan hasil ke dalam dictionary.
    """

    print(f"[INFO] Membaca dokumen PDF: {pdf_path}")
    doc = fitz.open(pdf_path)

    # Pastikan PDF punya halaman
    if doc.page_count == 0:
        print("[ERROR] PDF kosong!")
        return {"error": "PDF tidak memiliki halaman"}

    # 🔹 Ambil halaman pertama (indeks 0)
    print("[INFO] Step 1: Ekstraksi teks halaman pertama...")
    page = doc.load_page(0)  # Halaman pertama
    text = page.get_text("text").strip()

    # Jika teks kosong, bisa gunakan OCR (jika kamu punya fungsi OCR)
    if not text:
        print("[INFO] Halaman pertama kosong → OCR dijalankan...")
        text = process_page_ocr(page)  # Pastikan fungsi ini ada

    # Tutup dokumen
    doc.close()

    # Simpan hasil dalam dictionary
    parsed_result = {
        "halaman": 1,
        "text": text
    }

    if debug:
        print("[DEBUG] Teks hasil ekstraksi:\n", text[:500])  # tampilkan sebagian teks

    return parsed_result


In [11]:
def run_ocr(image_input) -> str:
    """
    Jalankan OCR pada gambar (numpy array atau path).
    - image_input: bisa berupa numpy array atau path ke file gambar
    - return: string gabungan teks hasil OCR
    """
    all_texts = []

    try:
        # Deteksi tipe input: path atau numpy array
        if isinstance(image_input, str):
            img = Image.open(image_input).convert("RGB")
            img_np = np.array(img)
        elif isinstance(image_input, np.ndarray):
            img_np = image_input
        else:
            raise ValueError("Input harus berupa path string atau numpy array.")

        # Jalankan OCR
        result = ocr_model.ocr(img_np)
        texts, scores = extract_texts_and_scores(result)
        all_texts.extend(texts)
        # print (all_texts)

    except Exception as e:
        print(f"[ERROR] OCR gagal: {e}")

    return " ".join(all_texts)

In [15]:
from paddleocr import PaddleOCR
import numpy as np
from PIL import Image

ocr_model = PaddleOCR(use_angle_cls=True, lang='en')
def extract_texts_and_scores(result):
    """Ekstraksi teks dan skor dari output PaddleOCR"""
    texts, scores = [], []
    if not result:
        return texts, scores

    first = result[0]
    if isinstance(first, dict):
        if 'rec_texts' in first:
            return first.get('rec_texts', []), first.get('rec_scores', [])
        if 'rec_res' in first:
            for item in first['rec_res']:
                if isinstance(item, (list, tuple)) and len(item) >= 2:
                    texts.append(item[0])
                    scores.append(item[1])
            return texts, scores

    # fallback jika struktur berbeda
    page_data = first if isinstance(first, (list, tuple)) else result

    def find_str(obj):
        if isinstance(obj, str):
            return obj
        if isinstance(obj, (list, tuple)):
            for e in obj:
                s = find_str(e)
                if s:
                    return s
        if isinstance(obj, dict):
            for v in obj.values():
                s = find_str(v)
                if s:
                    return s
        return None

    def find_num(obj):
        if isinstance(obj, (float, int)):
            return float(obj)
        if isinstance(obj, (list, tuple)):
            for e in obj:
                n = find_num(e)
                if n is not None:
                    return n
        if isinstance(obj, dict):
            for v in obj.values():
                n = find_num(v)
                if n is not None:
                    return n
        return None

    for line in page_data:
        text = find_str(line)
        score = find_num(line)
        texts.append(text or "")
        scores.append(score or 0.0)

    return texts, scores

C:\Users\Matt\AppData\Local\Temp\ipykernel_6208\2992479894.py:5: DeprecationWarning: The parameter `use_angle_cls` has been deprecated and will be removed in the future. Please use `use_textline_orientation` instead.
  ocr_model = PaddleOCR(use_angle_cls=True, lang='en')
Creating model: ('PP-LCNet_x1_0_doc_ori', None)
Using official model (PP-LCNet_x1_0_doc_ori), the model files will be automatically downloaded and saved in C:\Users\Matt\.paddlex\official_models.


Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Creating model: ('UVDoc', None)
The model(UVDoc) is not supported to run in MKLDNN mode! Using `paddle` instead!
Using official model (UVDoc), the model files will be automatically downloaded and saved in C:\Users\Matt\.paddlex\official_models.


Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Creating model: ('PP-LCNet_x1_0_textline_ori', None)
Using official model (PP-LCNet_x1_0_textline_ori), the model files will be automatically downloaded and saved in C:\Users\Matt\.paddlex\official_models.


Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Creating model: ('PP-OCRv5_server_det', None)
Using official model (PP-OCRv5_server_det), the model files will be automatically downloaded and saved in C:\Users\Matt\.paddlex\official_models.


Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Creating model: ('PP-OCRv5_server_rec', None)
Using official model (PP-OCRv5_server_rec), the model files will be automatically downloaded and saved in C:\Users\Matt\.paddlex\official_models.


Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

In [13]:
def process_page_ocr(page) -> str:
    """
    Proses OCR pada satu halaman PDF:
    - Render halaman ke gambar (numpy array)
    - Kirim ke OCR
    - Kembalikan teks hasilnya
    """
    try:
        pix = page.get_pixmap()
        mode = "RGBA" if pix.alpha else "RGB"
        img = Image.frombytes(mode, [pix.width, pix.height], pix.samples)
        img_np = np.array(img)

        text = run_ocr(img_np)
        return text.strip()

    except Exception as e:
        print(f"[ERROR] Gagal OCR halaman: {e}")
        return ""

In [27]:
file_path = "input/pdf/form checklist maintenance remote wireless.pdf"
all_teks = read_pdf(file_path, debug=True)

# Pastikan hasil bukan error
if "text" in all_teks:
    teks_halaman = all_teks["text"]

    # Simpan ke file .txt
    output_path = "output/form_checklist.txt"
    with open(output_path, "w", encoding="utf-8") as f:
        f.write(teks_halaman)

    print(f"[INFO] Teks berhasil disimpan ke: {output_path}")
else:
    print("[ERROR] Tidak ada teks yang bisa disimpan!")


[INFO] Membaca dokumen PDF: input/pdf/form checklist maintenance remote wireless.pdf
[INFO] Step 1: Ekstraksi teks halaman pertama...
[INFO] Halaman pertama kosong → OCR dijalankan...


C:\Users\Matt\AppData\Local\Temp\ipykernel_6208\3010705419.py:20: DeprecationWarning: Please use `predict` instead.
  result = ocr_model.ocr(img_np)


[DEBUG] Teks hasil ekstraksi:
 lintasarta FORM CHECKLIST MAINTENANCE REMOTE WIRELESS DATA REMOTE Nama Pelanggan ：BANKNEGARAINDONESIA1946(PERSERO) Contact Person Nomor Jaringan ：2021242440 ：Pak Arfan Nomor Telepon Alamat ：+62812-9024-5887 :JL.CUTNYAKDIENKALIMALANGUJUNG Kota : Propinsi No. SPK : ：083806/WO-LA/2021 Tanggal ：21-Jun-202118:00 Jam Perintah : 21-Jum-2021 Jam Persiapan ：21-Jun-2021 Jam Berangkat 18:00 ：21Jun-2021 Jam Tiba Di Lokasi : 21-Jun-2021 18:08 18:08 19:42 Jam Mulai Kerja ：21-Jum-2021 Jam Selesai Kerja 19:42 2
[INFO] Teks berhasil disimpan ke: output/form_checklist.txt
